# Flood ground-truth — explore

Two complementary flood layers, loaded, summarized, and merged into one
harmonized GeoDataFrame for pairing with the GOES imagery:

1. **groundsource** (`data/raw/groundsource_2026.parquet`) — observed flood
   *extents* (polygons) with a start/end date. ~2.65M rows, global.
2. **flood warnings** (`data/flood_warnings/flood_warnings_conus.parquet`) — NWS
   forecaster-issued **Flash Flood (FF)** + **Areal Flood (FA)** *Warning*
   polygons, CONUS 2019–2026, built by `src/download_flood_data.py`.

They differ in nature — observed extent vs. issued warning — so we keep a
`source` column rather than blending them blindly.

In [ ]:
from datetime import date
from pathlib import Path

import geopandas as gpd
import pandas as pd

pd.set_option("display.max_columns", None)

# repo root (works from notebooks/explore/, notebooks/, or the root itself)
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

GROUNDSOURCE = ROOT / "data/raw/groundsource_2026.parquet"
WARNINGS = ROOT / "data/flood_warnings/flood_warnings_conus.parquet"

# CONUS lon/lat box + study window (matches the warnings coverage)
CONUS = (-125, -66, 24, 49)
START = "2019-01-01"

## 1. Load

In [ ]:
# --- groundsource: observed flood extents ---
gs = gpd.read_parquet(
    GROUNDSOURCE,
    columns=["uuid", "area_km2", "start_date", "end_date", "geometry"],
)
gs["start_date"] = pd.to_datetime(gs["start_date"])
gs["end_date"] = pd.to_datetime(gs["end_date"])

# restrict to the CONUS + 2019-on study window (aligns with the warnings)
gs = gs.cx[CONUS[0]:CONUS[1], CONUS[2]:CONUS[3]]
gs = gs[gs["start_date"] >= START].copy()
print(f"groundsource (CONUS, >= {START}): {len(gs):,} rows")
gs.head(3)

In [ ]:
# --- NWS flood warnings (already CONUS 2019-2026) ---
fw = gpd.read_parquet(WARNINGS)
# IEM timestamps are ISO 'Z' (UTC); make them tz-naive UTC to match groundsource
fw["issue"] = pd.to_datetime(fw["issue"], utc=True).dt.tz_localize(None)
fw["expire"] = pd.to_datetime(fw["expire"], utc=True).dt.tz_localize(None)
print(f"flood warnings: {len(fw):,} rows")
fw.head(3)

## 2. Summarize

In [ ]:
def summarize(gdf, name, id_col, start_col, end_col, area_col):
    """One-line-per-stat summary of a flood GeoDataFrame."""
    print(f"=== {name} ===")
    print(f"rows            : {len(gdf):,}")
    print(f"unique ids      : {gdf[id_col].nunique():,}")
    print(f"date range      : {gdf[start_col].min():%Y-%m-%d} "
          f"-> {gdf[end_col].max():%Y-%m-%d}")
    dur = (gdf[end_col] - gdf[start_col]).dt.days
    print(f"duration (days) : median {dur.median():.0f}, mean {dur.mean():.1f}, "
          f"max {dur.max():.0f}")
    a = gdf[area_col]
    print(f"area_km2        : sum {a.sum():,.0f}, median {a.median():.2f}, "
          f"mean {a.mean():.1f}, max {a.max():,.0f}")
    print(f"geometry types  : {gdf.geometry.geom_type.value_counts().to_dict()}")
    print(f"bounds (lon/lat): {gdf.total_bounds.round(2).tolist()}")
    print()


summarize(gs, "groundsource (observed extents)", "uuid",
          "start_date", "end_date", "area_km2")
summarize(fw, "NWS flood warnings", "key", "issue", "expire", "area_iem")

In [ ]:
# warnings: breakdown by phenomena, year, and polygon source
print("by phenomena:", fw.phenomena.value_counts().to_dict())
print("polygon source:", fw.polygon_source.value_counts().to_dict())
fw.assign(year=fw.issue.dt.year).groupby(["year", "phenomena"]).size().unstack(fill_value=0)

In [ ]:
# groundsource: events per year (by start_date)
gs.assign(year=gs.start_date.dt.year).groupby("year").size()

## 3. Harmonized unified frame

One schema across both sources so they can be filtered, joined to GOES dates, and
overlaid together. Columns:

| column | groundsource | flood warning |
|---|---|---|
| `source` | `"groundsource"` | `"ff_warning"` / `"fa_warning"` |
| `event_id` | `uuid` | `key` (`year_wfo_ph_etn`) |
| `phenomena` | `"OBS"` | `FF` / `FA` |
| `issue_date` | `start_date` | `issue` |
| `expire_date` | `end_date` | `expire` |
| `area_km2` | `area_km2` | `area_iem` |
| `geometry` | polygon | polygon |

In [ ]:
COLS = ["source", "event_id", "phenomena", "issue_date", "expire_date",
        "area_km2", "geometry"]

gs_u = gpd.GeoDataFrame({
    "source": "groundsource",
    "event_id": gs["uuid"].astype(str),
    "phenomena": "OBS",
    "issue_date": gs["start_date"],
    "expire_date": gs["end_date"],
    "area_km2": gs["area_km2"],
    "geometry": gs.geometry,
}, crs=gs.crs)[COLS]

fw_u = gpd.GeoDataFrame({
    "source": fw["phenomena"].map({"FF": "ff_warning", "FA": "fa_warning"}),
    "event_id": fw["key"],
    "phenomena": fw["phenomena"],
    "issue_date": fw["issue"],
    "expire_date": fw["expire"],
    "area_km2": fw["area_iem"],
    "geometry": fw.geometry,
}, crs=fw.crs)[COLS]

floods = gpd.GeoDataFrame(
    pd.concat([gs_u, fw_u], ignore_index=True), crs="EPSG:4326"
)
floods["issue_day"] = floods["issue_date"].dt.normalize()   # date key for GOES join
print(f"unified: {len(floods):,} rows")
print("by source:", floods.source.value_counts().to_dict())
floods.head(3)

### Persist the unified frame

Save to `data/flood_warnings/floods_unified.parquet` so other notebooks (e.g.
`clouds_vs_floods.ipynb`) can overlay it without rebuilding.

In [ ]:
UNIFIED_OUT = ROOT / "data/flood_warnings/floods_unified.parquet"
floods.to_parquet(UNIFIED_OUT)
print(f"wrote {UNIFIED_OUT}  ({len(floods):,} rows)")

In [ ]:
# sanity: schema, null geometry, date span
print(floods.dtypes)
print("\nnull/empty geometry:", int(floods.geometry.isna().sum()),
      "/", int(floods.geometry.is_empty.sum()))
print("issue_date span:", floods.issue_date.min(), "->", floods.issue_date.max())
floods.describe(include="all").T

### Optional: overlay a sample on a map

`.explore()` puts a sample of both sources on one interactive map (warnings vs.
observed extents by colour).

In [ ]:
sample = floods.sample(10000, random_state=900)
sample.explore(
    column="source",
    tooltip=["source", "phenomena", "issue_date", "expire_date", "area_km2"],
    popup=True, cmap="viridis", style_kwds={"fillOpacity": 0.4},
)

## 4. Data inventory

One table of everything assembled so far: the two ground-truth layers loaded
above, plus what's on `/mnt/disk1` from the satellite downloaders (GOES ABI
images from `src/download_goes.py`, GLM flash day-parquets from
`src/download_glm.py` — the latter still downloading).

In [ ]:
# ---------------------------------------------------------------------------
# Data inventory — ground truth (this notebook) vs. satellite data on disk.
# ---------------------------------------------------------------------------
GOES_DIR = Path("/mnt/disk1/goes-data")
GLM_DIR = Path("/mnt/disk1/glm-data")
GLM_RANGE = (date(2019, 1, 1), date(2026, 2, 28))   # download_glm.py defaults

goes_files = list(GOES_DIR.glob("GOES*/*/*/*/*.nc"))
goes_days = {p.parent for p in goes_files}
glm_built = len(list(GLM_DIR.glob("*/glm_flashes_*.parquet")))
glm_total = (GLM_RANGE[1] - GLM_RANGE[0]).days + 1

inventory = pd.DataFrame([
    ("NWS flood warnings (FF + FA)", len(fw),
     f"by phenomena: {fw.phenomena.value_counts().to_dict()}"),
    ("groundsource flood events", len(gs),
     f"observed extents, CONUS, >= {START}"),
    ("GOES images downloaded", len(goes_files),
     f"{len(goes_days):,} days x 6 daytime frames (16-21 UTC)"),
    ("GLM flash days to download", glm_total,
     f"{glm_built:,} day-parquets built so far ({glm_built / glm_total:.0%}); "
     "~4,320 raw files/day -> 1 parquet/day"),
], columns=["dataset", "count", "detail"])
inventory.style.format({"count": "{:,}"}).hide(axis="index")

## 5. Do NWS warnings verify against groundsource?

Treat each **warning as a prediction** and groundsource as the **observation**:
for every warning, are there groundsource reports **spatially inside the warning
polygon** whose dates overlap the warning's active window? This matters for
training: it tells us how much the two candidate label sources agree, and which
to trust for what.

Matching rule: polygon **intersects** + date overlap of
`[start_date, end_date]` with `[issue_day, expire + 2 days]` — groundsource
extents are day-resolution and a flood's `start_date` can lag the rain, so a
2-day reporting-lag tolerance is allowed past expiry.

In [ ]:
# ---------------------------------------------------------------------------
# Match every warning with the groundsource obs inside it (space AND time).
# ---------------------------------------------------------------------------
LAG = pd.Timedelta(days=2)          # reporting-lag tolerance past warning expiry

# spatial: every (obs, warning) polygon intersection (STRtree-indexed)
pairs = gpd.sjoin(
    gs[["uuid", "start_date", "end_date", "geometry"]],
    fw[["key", "phenomena", "issue", "expire", "geometry"]],
    predicate="intersects", how="inner",
)
# temporal: [start_date, end_date] overlaps [issue_day, expire + LAG]
pairs = pairs[(pairs["start_date"] <= pairs["expire"] + LAG)
              & (pairs["end_date"] >= pairs["issue"].dt.normalize())]
print(f"{len(pairs):,} (obs, warning) matches | "
      f"{pairs['key'].nunique():,} warnings verified by "
      f"{pairs['uuid'].nunique():,} distinct obs")

# per-warning: how many obs landed inside it?
warn_v = fw[["key", "phenomena", "issue", "area_iem", "states"]].copy()
warn_v["n_obs"] = warn_v["key"].map(pairs.groupby("key").size()).fillna(0).astype(int)
warn_v["verified"] = warn_v["n_obs"] > 0
warn_v.sort_values("n_obs", ascending=False).head(8)

In [ ]:
# ---------------------------------------------------------------------------
# Event-level classification report.
#   precision = warnings with >=1 obs inside (verification rate)
#   recall    = obs covered by >=1 warning of that type (detection rate)
# (different units on the two sides -> f1 is indicative, not strict)
# ---------------------------------------------------------------------------
def event_report(phenomena=None):
    w = warn_v if phenomena is None else warn_v[warn_v["phenomena"] == phenomena]
    p = pairs if phenomena is None else pairs[pairs["phenomena"] == phenomena]
    precision = w["verified"].mean()
    recall = p["uuid"].nunique() / len(gs)
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall else 0.0)
    return {"warnings": len(w), "verified": int(w["verified"].sum()),
            "precision (verif. rate)": precision,
            "obs covered": p["uuid"].nunique(), "obs total": len(gs),
            "recall (detection rate)": recall, "f1": f1}


report = pd.DataFrame({"FF (flash flood)": event_report("FF"),
                       "FA (areal flood)": event_report("FA"),
                       "any warning": event_report()}).T
display(report.style.format({"precision (verif. rate)": "{:.1%}",
                             "recall (detection rate)": "{:.1%}", "f1": "{:.2f}",
                             "warnings": "{:,}", "verified": "{:,}",
                             "obs covered": "{:,}", "obs total": "{:,}"}))

# verification rate by year — is agreement stable over time?
(warn_v.assign(year=warn_v["issue"].dt.year)
       .pivot_table(index="year", columns="phenomena", values="verified",
                    aggfunc="mean").round(3))

### The training-grid view: cell-day classification report

The model will be trained per **25-km cell per day**, so the agreement that
actually matters is at that granularity: `y_pred` = cell-day inside an active
warning, `y_true` = cell-day inside an active groundsource extent. A standard
`sklearn.metrics.classification_report` applies cleanly here (no mixed units),
over all `cells × days` in the common 2019 → groundsource-end window. No lag
tolerance — this measures the labels exactly as a model would see them.

In [ ]:
# ---------------------------------------------------------------------------
# Rasterize both sources onto (25-km cell, day) and compare as a classifier.
# ---------------------------------------------------------------------------
import numpy as np
from sklearn.metrics import classification_report

grid = gpd.read_parquet("/mnt/disk1/goes-data/aux/conus_grid_25km.parquet")
D0 = pd.Timestamp(START)
D1 = gs["end_date"].max().normalize()       # common coverage window
days = pd.date_range(D0, D1, freq="D")
cell_pos = {c: i for i, c in enumerate(grid["cell_id"])}
day_pos = {d: i for i, d in enumerate(days)}


def active_cell_days(gdf, start_col, end_col):
    """Unique flat (cell x day) positions covered by polygons while active."""
    j = gpd.sjoin(grid[["cell_id", "geometry"]], gdf[[start_col, end_col,
                                                      "geometry"]],
                  predicate="intersects", how="inner")
    s, e = j[start_col].dt.normalize(), j[end_col].dt.normalize()
    keep = (s <= D1) & (e >= D0)        # boolean mask (sjoin index has dups)
    j, s, e = j[keep], s[keep].clip(D0, D1), e[keep].clip(D0, D1)
    n = ((e - s).dt.days + 1).to_numpy()
    cell_idx = j["cell_id"].map(cell_pos).to_numpy().repeat(n)
    day0 = s.map(day_pos).to_numpy().repeat(n)
    offsets = np.concatenate([np.arange(k) for k in n])
    return np.unique(cell_idx.astype(np.int64) * len(days) + day0 + offsets)


obs_cd = active_cell_days(gs, "start_date", "end_date")
warn_cd = active_cell_days(fw, "issue", "expire")

N = len(grid) * len(days)
y_true = np.zeros(N, dtype=bool)
y_true[obs_cd] = True
y_pred = np.zeros(N, dtype=bool)
y_pred[warn_cd] = True
print(f"{len(grid):,} cells x {len(days):,} days = {N:,} cell-days | "
      f"observed flood: {y_true.sum():,} ({y_true.mean():.2%}) | "
      f"warned: {y_pred.sum():,} ({y_pred.mean():.2%})\n")
print(classification_report(y_true, y_pred,
                            target_names=["no flood", "flood"], digits=3))